In [5]:
import sys
print(sys.executable)
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

/Users/mac/Desktop/Readmission-risk-predictor/venv/bin/python


In [6]:
import pandas as pd
from src.config import RAW_DATA_DIR
df = pd.read_csv(RAW_DATA_DIR / "diabetes130us_openml.csv")


In [8]:
df.replace("?", pd.NA, inplace = True)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide.metformin,glipizide.metformin,glimepiride.pioglitazone,metformin.rosiglitazone,metformin.pioglitazone,change,diabetesMed,readmitted
0,2278392.0,8222157.0,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190.0,55629189.0,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410.0,86047875.0,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364.0,82442376.0,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680.0,42519267.0,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548.0,100162476.0,AfricanAmerican,Male,[70-80),NaN,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782.0,74694222.0,AfricanAmerican,Female,[80-90),NaN,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148.0,41088789.0,Caucasian,Male,[70-80),NaN,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166.0,31693671.0,Caucasian,Female,[80-90),NaN,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [9]:
df.isnull().sum().sort_values(ascending=False)

weight                      98569
max_glu_serum               96420
A1Cresult                   84748
medical_specialty           49949
payer_code                  40256
race                         2273
diag_3                       1423
diag_2                        358
diag_1                         21
encounter_id                    0
troglitazone                    0
tolbutamide                     0
pioglitazone                    0
rosiglitazone                   0
acarbose                        0
miglitol                        0
citoglipton                     0
tolazamide                      0
examide                         0
glipizide                       0
insulin                         0
glyburide.metformin             0
glipizide.metformin             0
glimepiride.pioglitazone        0
metformin.rosiglitazone         0
metformin.pioglitazone          0
change                          0
diabetesMed                     0
glyburide                       0
repaglinide   

In [10]:
df = df.drop(columns=[
    "weight",
    "max_glu_serum",
    "A1Cresult"
])

In [11]:
cat_cols = [
    "medical_specialty",
    "payer_code",
    "race",
    "diag_3",
    "diag_2",
    "diag_1"]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [12]:
df.isnull().sum().sort_values(ascending=False)

encounter_id                0
tolazamide                  0
acetohexamide               0
glipizide                   0
glyburide                   0
tolbutamide                 0
pioglitazone                0
rosiglitazone               0
acarbose                    0
miglitol                    0
troglitazone                0
examide                     0
chlorpropamide              0
citoglipton                 0
insulin                     0
glyburide.metformin         0
glipizide.metformin         0
glimepiride.pioglitazone    0
metformin.rosiglitazone     0
metformin.pioglitazone      0
change                      0
diabetesMed                 0
glimepiride                 0
nateglinide                 0
patient_nbr                 0
num_lab_procedures          0
race                        0
gender                      0
age                         0
admission_type_id           0
discharge_disposition_id    0
admission_source_id         0
time_in_hospital            0
payer_code

In [22]:
df = df.drop(columns=[
    "encounter_id",
    "patient_nbr",
    "examide",
    "citoglipton"
])

In [23]:
df.shape

(101766, 43)

In [41]:
df["diag1_num"] = df["diag_1"].str.split(".").str[0]
df["diag2_num"] = df["diag_2"].str.split(".").str[0]
df["diag3_num"] = df["diag_3"].str.split(".").str[0]

In [42]:
df["diag1_num"] = df["diag1_num"].str.replace("[^0-9]", "", regex = True)
df["diag2_num"] = df["diag2_num"].str.replace("[^0-9]", "", regex = True)
df["diag3_num"] = df["diag3_num"].str.replace("[^0-9]", "", regex = True)

In [43]:
df["diag1_num"] = pd.to_numeric(df["diag1_num"], errors="coerce")
df["diag2_num"] = pd.to_numeric(df["diag2_num"], errors="coerce")
df["diag3_num"] = pd.to_numeric(df["diag3_num"], errors="coerce")

In [44]:
def categorize_icd9(code):
    if pd.isna(code):
        return "unknown"
    elif 1 <= code <= 139:
        return "infectious"

    elif 140 <= code <= 239:
        return "neoplasms"

    elif 240 <= code <= 279:
        return "endocrine"

    elif 280 <= code <= 289:
        return "blood"

    elif 290 <= code <= 319:
        return "mental"

    elif 320 <= code <= 389:
        return "nervous"

    elif 390 <= code <= 459:
        return "circulatory"

    elif 460 <= code <= 519:
        return "respiratory"

    elif 520 <= code <= 579:
        return "digestive"

    elif 580 <= code <= 629:
        return "genitourinary"

    elif 630 <= code <= 679:
        return "pregnancy"

    elif 680 <= code <= 709:
        return "skin"

    elif 710 <= code <= 739:
        return "musculoskeletal"

    elif 740 <= code <= 759:
        return "congenital"

    elif 760 <= code <= 779:
        return "perinatal"

    elif 780 <= code <= 799:
        return "symptoms"

    elif 800 <= code <= 999:
        return "injury"

    else:
        return "other"

In [47]:
df["diag1_group"] = df["diag1_num"].apply(categorize_icd9)
df["diag2_group"] = df["diag2_num"].apply(categorize_icd9)
df["diag3_group"] = df["diag3_num"].apply(categorize_icd9)


In [46]:
df["diag1_group"].value_counts()

diag1_group
circulatory        30357
endocrine          11459
respiratory        10407
digestive           9208
symptoms            7636
injury              6975
genitourinary       5078
musculoskeletal     4957
infectious          4412
neoplasms           3433
skin                2530
mental              2262
nervous             1211
blood               1103
pregnancy            687
congenital            51
Name: count, dtype: int64

In [48]:
df["diag2_group"].value_counts()

diag2_group
circulatory        31365
endocrine          21375
respiratory        10251
genitourinary       7987
symptoms            4632
digestive           3962
infectious          3736
skin                3596
injury              3159
blood               2926
mental              2657
neoplasms           2547
musculoskeletal     1764
nervous             1286
pregnancy            415
congenital           108
Name: count, dtype: int64

In [49]:
df["diag3_group"].value_counts()

diag3_group
circulatory        29918
endocrine          27731
respiratory         6774
genitourinary       6327
infectious          5675
symptoms            4523
digestive           3572
injury              3190
mental              3136
blood               2490
skin                2488
musculoskeletal     1915
neoplasms           1856
nervous             1766
pregnancy            309
congenital            96
Name: count, dtype: int64

In [51]:
df.shape    

(101766, 49)

In [53]:
df = df.drop(columns=[
    "diag_1", "diag_2", "diag_3",
    "diag1_num", "diag2_num", "diag3_num"
])

In [58]:
df["gender"] = df["gender"].map({"Male":1, "Female":0})
df["change"] = df["change"].map({"Ch":1, "No":0})
df["diabetesMed"] = df["diabetesMed"].map({"Yes": 1, "No":0})


In [60]:
age_map = {
    '[0-10)': 0, '[10-20)':1 , '[20-30)':2, '[30-40)':3,
    '[40-50)': 4, '[50-60)':5 , '[60-70)':6, '[70-80)':7,
    '[80-90)': 8, '[90-100)':9 
    }

df["age"] = df["age"].map(age_map)

In [61]:
med_map = {
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3,
}

In [64]:
med_cols = [
    'metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
    'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
    'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
    'insulin','glyburide.metformin','glipizide.metformin',
    'glimepiride.pioglitazone','metformin.rosiglitazone','metformin.pioglitazone'
]
for col in med_cols:
    df[col] = df[col].map(med_map)

In [65]:
df = pd.get_dummies(
    df,
    columns=[
        "race",
        "payer_code",
        "medical_specialty",
        "diag1_group",
        "diag2_group",
        "diag3_group"
    ],
    drop_first = True
)

In [66]:
df["readmitted"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)

In [72]:
X = df.drop("readmitted", axis =1)
y = df["readmitted"]

In [78]:
df.shape

(101763, 173)

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [74]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size = 0.5,
    random_state = 42,
    stratify=y_temp
)

In [75]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (71234, 172)
Validation: (15264, 172)
Test: (15265, 172)


In [76]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_val.to_csv("../data/processed/X_val.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [77]:
y_train.value_counts(normalize=True)
y_val.value_counts(normalize=True)
y_test.value_counts(normalize=True)

readmitted
0    0.888372
1    0.111628
Name: proportion, dtype: float64